# 02 Trump NLP: Topic Modelling
---
Working from `data/trump_clean_posts.csv` produced in notebook 01. Goal is to extract daily content-based features that can be merged with `data/trump_daily_features.csv` for the predictive model.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from bertopic import BERTopic

DATA_DIR = Path('../data')

posts = pd.read_csv(DATA_DIR / 'trump_clean_posts.csv', parse_dates=['post_date', 'created_at'])

days_with_posts = posts['post_date'].nunique()
total_days = (posts['post_date'].max() - posts['post_date'].min()).days + 1
days_no_posts = total_days - days_with_posts

print(f'Loaded {len(posts):,} posts across {days_with_posts} days ({days_no_posts} days in period with no text posts)')
posts.head(3)

/Users/simon/advanced_business_analytics/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loaded 1,875 posts across 145 days (3 days in period with no text posts)


,id,post_date,created_at,text,is_quote,has_link,favourites_count,reblogs_count,replies_count
0,115549907135940704,2025-11-14,2025-11-14 20:20:59.178000+00:00,I am pleased to nominate Megan Benton to serve...,False,False,15690,3153,466
1,115549914983720560,2025-11-14,2025-11-14 20:22:58.926000+00:00,It is my honor to nominate Justin Olson to ser...,False,False,17271,3538,1265
2,115551054294277072,2025-11-15,2025-11-15 01:12:43.425000+00:00,"Did Thomas Massie, sometimes referred to as Ra...",False,False,19076,4675,2984


---
## 1. Text preprocessing

In [2]:
import re

def clean_text(text):
    # Truth Social breaks URLs with spaces, e.g. "https:// domain.com/ path"
    # Remove the https:// anchor and any following slash-separated fragments
    text = re.sub(r'https?://\s*\S*', '', text)     # remove https://... 
    text = re.sub(r'\b\S+/\S+\b', '', text)          # remove remaining /path fragments
    text = re.sub(r'^RT:\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

posts['text_clean'] = posts['text'].apply(clean_text)
posts['clean_len']  = posts['text_clean'].str.len()

n_short = (posts['clean_len'] < 30).sum()
print(f'Posts too short after cleaning (<30 chars): {n_short}  — excluded from topic modelling')
print(f'Posts used for topic modelling            : {(posts["clean_len"] >= 30).sum()}')
print()
# Sample check
sample = posts[posts['text'].str.contains(r'https?://', regex=True, na=False)].iloc[0]
print('Before:', sample['text'][:200])
print('After :', sample['text_clean'][:200])

Posts too short after cleaning (<30 chars): 516  — excluded from topic modelling
Posts used for topic modelling            : 1359

Before: https:// americanrefugees.substack.com/ p/will-john-brennan-finally-get-his
After : 


---
## 2. Fit BERTopic

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

mask    = posts['clean_len'] >= 30
docs    = posts.loc[mask, 'text_clean'].tolist()
doc_idx = posts.loc[mask].index.tolist()

vectorizer    = CountVectorizer(stop_words='english', min_df=2, ngram_range=(1, 2))
umap_model    = UMAP(n_components=5, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=20, prediction_data=True)

topic_model = BERTopic(
    language='english',
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    min_topic_size=20,
    nr_topics='auto',
    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

n_topics   = len(set(topics)) - 1
n_outliers = sum(t == -1 for t in topics)
print(f'\nTopics found  : {n_topics}')
print(f'Outliers (-1) : {n_outliers}/{len(docs)} fitted posts ({n_outliers/len(docs)*100:.1f}%)')

2026-04-21 15:39:40,522 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

2026-04-21 15:39:47,902 - BERTopic - Embedding - Completed ✓


2026-04-21 15:39:47,903 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


2026-04-21 15:39:53,974 - BERTopic - Dimensionality - Completed ✓


...

2026-04-21 15:39:54,096 - BERTopic - Topic reduction - Reducing number of topics


2026-04-21 15:39:54,099 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-04-21 15:39:54,174 - BERTopic - Representation - Completed ✓


2026-04-21 15:39:54,175 - BERTopic - Topic reduction - Reduced number of topics from 17 to 14



Topics found  : 13
Outliers (-1) : 363/1359 fitted posts (26.7%)


In [4]:
topic_info = topic_model.get_topic_info()
topic_info[['Topic', 'Count', 'Name', 'Representation']]

,Topic,Count,Name,Representation
0,-1,363,-1_trump_president_great_country,"[trump, president, great, country, world, new,..."
1,0,267,0_state_secure_district_endorsement,"[state, secure, district, endorsement, total e..."
2,1,152,1_tariffs_states_united states_united,"[tariffs, states, united states, united, court..."
3,2,134,2_minnesota_ice_criminals_democrats,"[minnesota, ice, criminals, democrats, country..."
4,3,114,3_iran_president_donald_president donald,"[iran, president, donald, president donald, do..."
5,4,69,4_white house_white_center_trump,"[white house, white, center, trump, ballroom, ..."
6,5,46,5_president djt_djt_president_great,"[president djt, djt, president, great, fantast..."
7,6,43,6_epstein_fake_times_news,"[epstein, fake, times, news, york times, fake ..."
8,7,42,7_venezuela_president_oil_honduras,"[venezuela, president, oil, honduras, united s..."
9,8,31,8_state union_union_trump_trumps,"[state union, union, trump, trumps, address, s..."


---
## What we did and why

**Step 1 — Clean the text**
Before any modelling, we removed URLs, leftover web address fragments, and "RT:" prefixes from each post. This is important because BERTopic would otherwise treat URL words ("substack", "com", "https") as meaningful content and cluster posts around them instead of the actual message. Posts that had nothing left after cleaning (fewer than 30 characters) were excluded entirely — these were essentially just a shared link with no text.

**Step 2 — Find topics with BERTopic**
BERTopic reads all posts, converts them to numerical representations using a language model, and then groups similar posts together into topics. We set three key parameters:
- Minimum 20 posts per topic — avoids tiny one-off clusters with no daily signal
- Auto topic merging — similar topics are automatically combined
- Fixed random seed — ensures we get the same topics every time we re-run the notebook

**Step 3 — Why probabilities, not topic labels**
Rather than assigning each post a single topic ("this post is about tariffs"), BERTopic gives each post a probability score for every topic ("this post is 70% tariffs, 20% courts, 10% other"). We use these probabilities as our daily signal. This is better for two reasons: (1) posts often cover multiple themes at once, and (2) the output is always the same number of columns regardless of how many topics were found in a given run.

**Topics found**

| Topic | Keywords | Theme |
|---|---|---|
| 0 | state, district, secure, endorsement | Endorsements / appointments |
| 1 | tariffs, united states, court | Trade & economic policy |
| 2 | minnesota, ice, criminals | Immigration & border |
| 3 | iran, donald, president | Foreign policy — Middle East |
| 4 | white house, center, ballroom | White House events |
| 5 | president djt, great, wonderful | General presidential messaging |
| 6 | epstein, fake news, york times | Media & investigations |
| 7 | venezuela, oil, honduras | Latin America / energy |
| 8 | save america, act, voter id | Election integrity |
| 9 | state union, union, speech | State of the Union |
| 10 | judge, court, district | Judicial |
| 11 | peace, board, gaza | Middle East peace |
| 12 | water, mexico, storm, maryland | Environment / disasters |

Posts that do not fit any topic (general MAGA statements, congratulations, sports) end up as outliers and are not included in the probability output.

---
## 3. Convert topics to daily signals

**What we do:** For each post, BERTopic gives a probability score for every topic. We sum those scores across all posts in a day to get one number per topic per day.


We use `.sum()` rather than `.mean()` for two reasons:

1. **Captures intensity.** A day with 20 tariff posts is meaningfully different from a day with 2. Averaging would treat both the same if the topic mix is similar. Summing reflects how much total attention Trump devoted to a topic that day.

2. **Avoids perfect multicollinearity.** If we averaged, every day's row would sum to exactly 1 (because each post's probability vector sums to 1). That creates a linear dependency across all columns, which breaks most regression and ML models. With sums, different days have different total post counts, so the columns no longer sum to a constant.

**Trade-off to be aware of:** Because the sum depends on how many posts were made that day, high-activity days will naturally have higher absolute scores across all topics — not because the content changed, but simply because Trump posted more. The model may partly pick up "busy posting days" rather than pure topic focus. If this is a concern, total post count is already in the daily features and can be used to control for it.

**What it can be used for:** Each `topic_prob_X` column is a daily feature for the predictive model. It can serve as a signal for economic outcomes (e.g. tariff activity before market moves) and political resilience outcomes (e.g. how much Trump posts about fake news or election integrity on days when approval ratings shift).

In [5]:
# Normalise each post's probability vector to sum to 1, then round to clean decimals
# (raw HDBSCAN probs underflow to ~1e-308 instead of true 0)
probs_norm = np.round(probs / probs.sum(axis=1, keepdims=True), 4)

# Build a probability DataFrame — one row per post, one column per topic
prob_df = pd.DataFrame(probs_norm, index=doc_idx,
                       columns=[f'topic_prob_{i}' for i in range(probs_norm.shape[1])])
prob_df['post_date'] = posts.loc[doc_idx, 'post_date'].values

# Sum per day — gives total topic activity, captures both focus and volume
daily_probs     = prob_df.groupby('post_date').sum().round(2)
topic_prob_cols = daily_probs.columns.tolist()

# Merge onto original daily features — save to new file, original stays untouched
daily     = pd.read_csv(DATA_DIR / 'trump_daily_features.csv', parse_dates=['post_date'])
daily_nlp = daily.merge(daily_probs.reset_index(), on='post_date', how='left')
daily_nlp[topic_prob_cols] = daily_nlp[topic_prob_cols].fillna(0)

out_path = DATA_DIR / 'trump_daily_features_nlp.csv'
daily_nlp.to_csv(out_path, index=False)

print(f'Saved {len(daily_nlp)} rows × {len(daily_nlp.columns)} columns → {out_path.name}')
print(f'NLP columns added: {len(topic_prob_cols)}')
daily_nlp[['post_date'] + topic_prob_cols].head(5)

Saved 148 rows × 25 columns → trump_daily_features_nlp.csv
NLP columns added: 13


,post_date,topic_prob_0,topic_prob_1,topic_prob_2,topic_prob_3,topic_prob_4,topic_prob_5,topic_prob_6,topic_prob_7,topic_prob_8,topic_prob_9,topic_prob_10,topic_prob_11,topic_prob_12
0,2025-11-14,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.00,0.00
1,2025-11-15,1.23,0.14,0.65,0.14,0.19,0.37,0.92,0.13,0.27,0.31,0.20,0.24,0.21
2,2025-11-16,0.96,0.00,0.01,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.01,0.00
3,2025-11-17,5.09,0.14,0.53,0.14,0.17,0.30,1.39,0.13,0.25,0.23,1.19,0.24,0.20
4,2025-11-18,1.31,0.74,0.51,0.20,0.19,0.24,0.25,0.26,1.26,0.33,0.24,0.13,0.35


---
## 4. Spot-check: posts vs topic probabilities

For a few dates we pick the day with the highest probability for a given topic, print the actual posts, and check whether the content makes sense.

In [6]:
def inspect_day(date_str, daily_probs, posts, topic_prob_cols):
    date = pd.Timestamp(date_str)
    print(f'=== {date_str} ===')

    # Topic probabilities for this day — show top 5
    row = daily_probs.loc[date, topic_prob_cols].sort_values(ascending=False)
    topic_info_df = topic_model.get_topic_info().set_index('Topic')
    print('\nTop 5 topic probabilities:')
    for col, val in row.head(5).items():
        topic_num = int(col.split('_')[-1])
        name = topic_info_df.loc[topic_num, 'Name'] if topic_num in topic_info_df.index else 'n/a'
        print(f'  {col}: {val:.4f}  [{name}]')

    # Posts that day
    day_posts = posts[posts['post_date'] == date]['text'].reset_index(drop=True)
    print(f'\nPosts that day ({len(day_posts)} total):')
    for i, text in day_posts.items():
        print(f'\n  [{i+1}] {text[:250]}')
    print()

# Pick the peak day for 3 topics of interest
topics_to_check = {
    'Trade / tariffs  (topic_prob_1)':      'topic_prob_1',
    'Immigration      (topic_prob_2)':      'topic_prob_2',
    'Iran / foreign policy (topic_prob_3)': 'topic_prob_3',
}

for label, col in topics_to_check.items():
    peak_date = daily_probs[col].idxmax()
    print(f'\n{"="*60}')
    print(f'Checking: {label}')
    inspect_day(str(peak_date.date()), daily_probs, posts, topic_prob_cols)


Checking: Trade / tariffs  (topic_prob_1)
=== 2026-02-27 ===

Top 5 topic probabilities:
  topic_prob_0: 20.2200  [0_state_secure_district_endorsement]
  topic_prob_8: 14.3500  [8_state union_union_trump_trumps]
  topic_prob_2: 5.3700  [2_minnesota_ice_criminals_democrats]
  topic_prob_6: 3.8200  [6_epstein_fake_times_news]
  topic_prob_1: 3.8100  [1_tariffs_states_united states_united]

Posts that day (68 total):

  [1] Congrats to Newsmax. Big numbers! President DJT Newsmax Delivers 4M-Plus Viewers in Breakout State of Union Coverage:  https://www. newsmax.com/newsfront/newsmax- tv-rating-sotu/2026/02/26/id/1247600/

  [2] Mortgage rates drop to under 6% for first time since 2022: Freddie Mac:  https:// justthenews.com/nation/economy /mortgage-rates-drop-under-6-first-time-2022-freddie-mac

  [3] Trump draws clear midterm battle lines: Patriots vs. 'Hate-triots’:  https:// justthenews.com/government/whi te-house/midterm-battle-lines-drawn-patriots-vs-hateriots

  [4] As Georgia pros

---
## Note: Known limitations for modelling

**Look-ahead bias in topic fitting**
The topic model was fitted on all 1,359 posts at once — including posts that would fall in the test period of any future train/test split. In a strict forecasting setting, BERTopic should be fitted on training data only and then applied to the test period. We accepted this limitation because 148 days is too small to split and still get stable topics from the training portion alone. The topics are descriptive and stable (fixed random seed), but probability values for test-period days carry some bias.

**Lagging the features**
These topic signals are currently aligned same-day with any outcome variable. For economic or market outcomes this creates a reverse causality risk: Trump may be reacting to market moves in his posts, rather than his posts driving markets. Before using these features in a predictive model, consider lagging all `topic_prob_X` columns by at least 1 day so the features strictly precede the outcome. The right lag length depends on the target variable and should be decided in the modelling notebook.

---
## 5. Daily sentiment

**What it is:** Sentiment analysis scores each post on a scale from -1 (very negative/angry) to +1 (very positive/triumphant). We use VADER, a rule-based model built specifically for social media text — it understands things like CAPS, exclamation marks, and common slang without needing any training.

**Why it adds value on top of topics:** Topics tell us *what* Trump posts about. Sentiment tells us *how*. A tariff post that says "Tariffs are DESTROYING our enemies!!!" and one that says "We are carefully reviewing our tariff policy" describe the same topic but carry completely different signals. The tone of a post may matter as much as the subject.

**Features produced:**
- `sentiment_mean` — average daily tone (positive = confident/celebratory, negative = attacking/alarmed)
- `sentiment_std` — how much the tone varied within that day (high = mixed messaging)
- `sentiment_pct_negative` — share of posts with strong negative tone (compound < -0.5)

In [7]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

# Score each post — use text_clean (URLs removed) but CAPS and ! are preserved
posts['sentiment'] = posts['text_clean'].apply(
    lambda x: analyzer.polarity_scores(x)['compound'] if len(x) > 10 else np.nan
)

daily_sentiment = posts.groupby('post_date')['sentiment'].agg(
    sentiment_mean = 'mean',
    sentiment_std  = 'std',
    sentiment_pct_negative = lambda x: (x < -0.5).mean()
).round(4)

print(f'Sentiment features computed for {len(daily_sentiment)} days')
print(f'\nOverall mean sentiment: {daily_sentiment["sentiment_mean"].mean():.3f}')
print(f'Most negative day:      {daily_sentiment["sentiment_mean"].idxmin().date()}  ({daily_sentiment["sentiment_mean"].min():.3f})')
print(f'Most positive day:      {daily_sentiment["sentiment_mean"].idxmax().date()}  ({daily_sentiment["sentiment_mean"].max():.3f})')
daily_sentiment.head(5)

Sentiment features computed for 145 days

Overall mean sentiment: 0.320
Most negative day:      2026-02-14  (-0.986)
Most positive day:      2026-02-15  (0.994)


,sentiment_mean,sentiment_std,sentiment_pct_negative
post_date,,,
2025-11-14,0.9656,0.0317,0.0000
2025-11-15,0.1946,0.8277,0.2857
2025-11-16,0.6651,NaN,0.0000
2025-11-17,0.2066,0.5047,0.0800
2025-11-18,0.5108,0.4714,0.0000


---
## 6. Rhetorical intensity

**What it is:** Simple text statistics computed directly from the raw post text — no model required. We measure two things: how much Trump uses ALL-CAPS words (a proxy for shouting/urgency) and how many exclamation marks he uses per post.

**Why it adds value:** These features capture Trump's emotional register — how agitated or emphatic he is — completely independently of what he's talking about or whether the tone is positive or negative. A calm tariff post looks very different from "TARIFFS ARE THE GREATEST WEAPON EVER USED!!!" even though both might score similarly on sentiment. High-intensity days may signal a different kind of attention or reaction from followers.

**Features produced:**
- `caps_ratio` — average share of words in ALL CAPS per post (e.g. 0.15 = 15% of words are capitalised)
- `exclaim_per_post` — average number of exclamation marks per post

In [8]:
def caps_ratio(text):
    words = text.split()
    if not words:
        return 0.0
    caps_words = [w for w in words if w.isupper() and len(w) > 1]
    return len(caps_words) / len(words)

# Compute on original text to preserve authentic casing
posts['caps_ratio']      = posts['text'].apply(caps_ratio)
posts['exclaim_count']   = posts['text'].str.count('!')

daily_intensity = posts.groupby('post_date').agg(
    caps_ratio      = ('caps_ratio',    'mean'),
    exclaim_per_post = ('exclaim_count', 'mean')
).round(4)

print(f'Intensity features computed for {len(daily_intensity)} days')
print(f'\nMean caps ratio:        {daily_intensity["caps_ratio"].mean():.3f}')
print(f'Mean exclaims per post: {daily_intensity["exclaim_per_post"].mean():.2f}')
print(f'\nHighest intensity day:  {daily_intensity["caps_ratio"].idxmax().date()}  (caps_ratio={daily_intensity["caps_ratio"].max():.3f})')
daily_intensity.head(5)

Intensity features computed for 145 days

Mean caps ratio:        0.142
Mean exclaims per post: 1.50

Highest intensity day:  2026-01-05  (caps_ratio=0.462)


,caps_ratio,exclaim_per_post
post_date,,
2025-11-14,0.0228,1.5000
2025-11-15,0.2186,2.2857
2025-11-16,0.0395,2.5000
2025-11-17,0.0364,1.0800
2025-11-18,0.1827,1.3333


---
## 7. Merge all signals and re-evaluate

We now have three sets of NLP-derived daily features. Before merging into the final dataset, it is worth comparing what each one captures and whether it is worth including.

| Signal set | What it captures | # features | Main risk |
|---|---|---|---|
| Topic activity | *What* Trump posts about (tariffs, immigration, Iran…) | 13 | Sparse topics, look-ahead bias |
| Sentiment | *How positive or negative* the tone is | 3 | VADER is generic, not politics-tuned |
| Rhetorical intensity | *How agitated or emphatic* the language is | 2 | Blunt proxy, may correlate with post volume |

**Verdict on each:**

*Topic activity* is the richest signal but most of the 13 columns are noise for economic outcomes. For political resilience, more of them are relevant. Keep all 13 for now and let the model sort out which matter — use regularisation (LASSO/Ridge) to suppress irrelevant ones.

*Sentiment* is a clean, low-dimensional signal (3 columns) that is genuinely orthogonal to topics. Well worth including. Main caveat: VADER was not built for political speech and may misread Trump's rhetorical style (e.g. sarcasm, coded language).

*Rhetorical intensity* is the simplest of the three (2 columns, no model). Caps ratio and exclamation density are crude but they are available for every post, require no assumptions, and capture something neither topics nor sentiment fully captures — the *agitation level* of a given day. Worth including given the low cost.

**Recommendation: include all three.** Total NLP features added = 18. With 148 observations this is tight, so the modelling notebook should use regularisation and consider dropping the sparsest topic columns (8, 9, 10, 11, 12) if the model shows signs of overfitting.

In [9]:
# Merge all three signal sets onto daily features
daily     = pd.read_csv(DATA_DIR / 'trump_daily_features.csv', parse_dates=['post_date'])
daily_nlp = (daily
    .merge(daily_probs.reset_index(),     on='post_date', how='left')
    .merge(daily_sentiment.reset_index(), on='post_date', how='left')
    .merge(daily_intensity.reset_index(), on='post_date', how='left')
)

# Fill NaN for days with no text posts
nlp_cols = topic_prob_cols + ['sentiment_mean', 'sentiment_std', 'sentiment_pct_negative',
                               'caps_ratio', 'exclaim_per_post']
daily_nlp[nlp_cols] = daily_nlp[nlp_cols].fillna(0)

out_path = DATA_DIR / 'trump_daily_features_nlp.csv'
daily_nlp.to_csv(out_path, index=False)

print(f'Saved {len(daily_nlp)} rows × {len(daily_nlp.columns)} columns → {out_path.name}')
print(f'\nNLP features added:')
print(f'  Topic activity    : {len(topic_prob_cols)} columns')
print(f'  Sentiment         : 3 columns')
print(f'  Rhetorical intensity: 2 columns')
print(f'  Total NLP         : {len(nlp_cols)} columns')
daily_nlp[['post_date'] + nlp_cols].tail(5)

Saved 148 rows × 30 columns → trump_daily_features_nlp.csv

NLP features added:
  Topic activity    : 13 columns
  Sentiment         : 3 columns
  Rhetorical intensity: 2 columns
  Total NLP         : 18 columns


,post_date,topic_prob_0,topic_prob_1,topic_prob_2,topic_prob_3,topic_prob_4,topic_prob_5,topic_prob_6,topic_prob_7,topic_prob_8,topic_prob_9,topic_prob_10,topic_prob_11,topic_prob_12,sentiment_mean,sentiment_std,sentiment_pct_negative,caps_ratio,exclaim_per_post
143,2026-04-06,1.07,0.29,0.79,0.24,0.28,0.89,0.41,0.23,0.69,0.33,0.40,0.19,1.20,0.0823,0.5740,0.100,0.1120,1.2000
144,2026-04-07,28.10,0.45,1.00,1.54,0.51,0.67,0.52,0.43,0.59,0.49,0.64,0.53,0.53,0.8760,0.3636,0.027,0.1517,2.0811
145,2026-04-08,0.70,0.32,0.46,3.30,0.31,0.37,0.28,0.47,0.41,0.22,1.62,0.18,0.36,0.1397,0.7393,0.300,0.1410,1.7000
146,2026-04-09,0.45,0.25,0.72,4.08,1.26,0.33,1.37,0.43,0.35,0.29,1.01,1.15,0.32,-0.0562,0.8569,0.500,0.0909,1.9167
147,2026-04-10,0.20,0.19,0.22,0.13,0.19,0.19,0.14,0.93,0.23,0.12,0.21,0.07,0.19,0.0593,0.1027,0.000,0.0000,0.0000
